# Chapter 14 &mdash; Why Study Impossibility Results?

**Concept 1 of the Chapter 14 decomposition:** *Why Study Impossibility Results?*

Knowing what a machine <i>cannot</i> do is as informative as knowing what it can.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-Why-Impossibility-Results/Concept-Why-Impossibility-Results.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Impossibility results are not pessimism; they are **engineering information**.

* They tell you **when to stop looking.** No amount of cleverness yields a perfect
  static analyser for "does this program crash", so tool builders aim for *sound but
  incomplete* or *complete but unsound* instead.
* They **redirect effort.** Type systems, model checkers, and linters are all answers
  to "the general question is undecidable, so what restricted question is not?"
* They are **robust.** Unlike performance claims, they do not expire when hardware
  improves.

The pattern to internalise: *the general problem is undecidable; a decidable
approximation is what ships*.

## 2. Definitions

### A decidable question and its undecidable big brother

In [ ]:
def re_dfa_emptiness(D):
    # DECIDABLE: is L(D) empty?  Reachability from q0 to any final state.
    seen, frontier = {D["q0"]}, {D["q0"]}
    while frontier:
        nxt = {step_dfa(D, q, a) for q in frontier for a in D["Sigma"]} - seen
        seen |= nxt; frontier = nxt
    return not (seen & D["F"])

### The approximation pattern, spelled out

In [ ]:
STANCES = [("sound, incomplete", "never wrong when it says 'safe'; may refuse to decide",
            "type systems, most static analysers"),
           ("complete, unsound", "finds every bug; also reports non-bugs",
            "aggressive linters, some fuzz triage"),
           ("bounded",           "decides correctly up to a size or depth limit",
            "bounded model checking, SAT-based tools")]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch13&nbsp;13.&nbsp;The Compact ID Notation $aqb$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-Compact-ID-Notation/Concept-Compact-ID-Notation.ipynb) &nbsp;&middot;&nbsp; [**Chapter 14** index](https://github.com/ganeshutah/Jove/blob/master/Chapter14/README.md) &nbsp;&middot;&nbsp; [Ch14&nbsp;2.&nbsp;Procedure vs. Algorithm, and the First Impossibility Result](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-Procedure-Vs-Algorithm/Concept-Procedure-Vs-Algorithm.ipynb)&nbsp;&rarr;

---

## 3. Tests

DFA emptiness **is** decidable, and the algorithm always terminates.

In [ ]:
Empty = md2mc('''DFA
I : 0 | 1 -> I
''')
NotEmpty = md2mc('''DFA
IF : 0 | 1 -> IF
''')
print("L(Empty)    is empty?", re_dfa_emptiness(Empty))
print("L(NotEmpty) is empty?", re_dfa_emptiness(NotEmpty))
assert re_dfa_emptiness(Empty) and not re_dfa_emptiness(NotEmpty)

It terminates on **every** DFA, which is what 'decidable' means.

In [ ]:
import random
for trial in range(20):
    n = random.randint(1, 6)
    names = ['I'] + ['S%d' % i for i in range(1, n)]
    fin = random.sample(names, random.randint(0, n))
    lines = ['DFA']
    for q in names:
        for a in '01':
            t = random.choice(names)
            lines.append('%s : %s -> %s' % (('F' + q if q in fin and q != 'I'
                                             else ('IF' if q == 'I' and q in fin else q)),
                                            a, ('F' + t if t in fin and t != 'I'
                                                else ('IF' if t == 'I' and t in fin else t))))
    D = md2mc('\n'.join(lines))
    re_dfa_emptiness(D)      # must simply return, every time
print("emptiness decided for 20 random DFA, no timeouts, no fuel")

The same question about **programs** is not decidable &mdash; so tools approximate.

In [ ]:
print("%-20s %-52s %s" % ("stance", "guarantee", "examples"))
for a, b, d in STANCES:
    print("%-20s %-52s %s" % (a, b, d))

What impossibility buys you, concretely.

In [ ]:
USES = ["stop searching for the perfect tool -- it does not exist",
        "choose your unsoundness deliberately rather than by accident",
        "explain to a manager why 'just detect all bugs' is not a backlog item",
        "recognise a reduction when a new problem is the old one in disguise"]
for u in USES: print("  *", u)

## 4. Exercises


1. Name a tool you use daily that is deliberately incomplete. What does it give up?
2. Is "does this regular expression match anything?" decidable? Why?
3. Which of the three stances would you pick for a compiler warning? For a verifier?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter14/Concept-Why-Impossibility-Results')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')